# NiyamTrace-X Q1 Wave 4 / Merger
## Merge parallel Q1 experiments into paper-ready artifacts

Run after Experiments A–C complete. Experiment D is optional until human annotations are finished.

Upload:
- `NTX_Q1_W4_A_AGENTDOJO_RESULTS.zip`
- `NTX_Q1_W4_B_UNSEEN_REDTEAM_RESULTS.zip`
- `NTX_Q1_W4_C_BROKER_V3_RESULTS.zip`
- optionally `NTX_Q1_W4_D_HUMAN_BENCHMARK_RESULTS.zip`

The merger verifies manifests, refuses to treat a smoke-generated red-team run as paper evidence,
creates a claim/evidence matrix, manuscript tables and a combined ZIP.

In [ ]:
from pathlib import Path
import json, zipfile, shutil, os
import numpy as np, pandas as pd

BASE=Path("/content") if Path("/content").exists() else Path("/mnt/data")
ROOT=BASE/"NTX_Q1_W4_MERGED"
if ROOT.exists():shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

required=[
 "NTX_Q1_W4_A_AGENTDOJO_RESULTS.zip",
 "NTX_Q1_W4_B_UNSEEN_REDTEAM_RESULTS.zip",
 "NTX_Q1_W4_C_BROKER_V3_RESULTS.zip",
]
missing=[x for x in required if not (BASE/x).exists()]
if missing:
    try:
        from google.colab import files
        print("Upload required ZIPs:",missing)
        up=files.upload()
        for name,data in up.items():(BASE/name).write_bytes(data)
    except Exception:pass
missing=[x for x in required if not (BASE/x).exists()]
assert not missing, "Missing: "+str(missing)

In [ ]:
# Extract and validate
dirs=[]
for name in required:
    d=ROOT/name.replace(".zip","")
    d.mkdir()
    with zipfile.ZipFile(BASE/name) as z:
        assert z.testzip() is None
        z.extractall(d)
    dirs.append(d)

def one(root,name):
    hits=list(Path(root).rglob(name))
    assert hits, name+" missing in "+str(root)
    return hits[0]

mans=[json.loads(one(d,"MANIFEST.json").read_text()) for d in dirs]
display(pd.DataFrame(mans))

red=[m for m in mans if "_B_" in m.get("experiment","")]
assert red and red[0].get("source")=="FROZEN_HOLDOUT", "Red-team run is not based on the frozen holdout; refusing paper merge."

In [ ]:
# Headline tables
a=pd.read_csv(one(dirs[0],"method_summary.csv"))
b=pd.read_csv(one(dirs[1],"overall_metrics.csv"))
c=pd.read_csv(one(dirs[2],"overall_metrics.csv"))
lat=pd.read_csv(one(dirs[2],"latency_summary.csv"))

ntx=a[a.method=="SET_INTERSECTION"].iloc[0]
rb=b.iloc[0]
br=c.iloc[0]

claims=pd.DataFrame([
 {"claim":"External stateful authorization replay",
  "evidence":str(int(ntx.attack_n))+" mutated AgentDojo action cases; unsafe commit "+format(ntx.unsafe_commit_rate,".2%"),
  "boundary":"Authorization replay, not native AgentDojo LLM leaderboard score."},
 {"claim":"Frozen v3 generalization beyond the original patch family",
  "evidence":str(int(rb.attack_detected))+"/"+str(int(rb.attack_n))+" attacks detected; clean FPR "+format(rb.clean_fpr,".2%"),
  "boundary":"Only frozen post-v3 attack families tested in Experiment B."},
 {"claim":"Transaction-bound reference broker resists tested faults/races",
  "evidence":str(int(br.unsafe_commits))+" unsafe commits / "+str(int(br.invalid_or_fault_trials))+" invalid/fault trials; race safe "+format(br.race_safe_rate,".2%"),
  "boundary":"Reference SQLite broker, not production-complete mediation."},
])
claims.to_csv(ROOT/"Q1_claim_evidence_matrix.csv",index=False)
display(claims)

In [ ]:
# Paper table + text
tab=a[["method","unsafe_commit_rate","attack_interception","clean_completion","decision_accuracy","median_gate_us","p95_gate_us"]].copy()
tab.to_csv(ROOT/"Table_external_authorization.csv",index=False)

tex=[
 "\\begin{table}[t]","\\centering",
 "\\caption{External stateful authorization replay on AgentDojo task environments.}",
 "\\begin{tabular}{lrrrr}","\\hline",
 "Method & Unsafe commit & Intercept & Clean completion & Accuracy \\\\","\\hline"
]
for _,r in tab.iterrows():
    method=str(r.method).replace("_","\\_")
    tex.append(method+" & "+format(r.unsafe_commit_rate,".3f")+" & "+format(r.attack_interception,".3f")+
               " & "+format(r.clean_completion,".3f")+" & "+format(r.decision_accuracy,".3f")+" \\\\")
tex += ["\\hline","\\end{tabular}","\\end{table}"]
(ROOT/"Table_external_authorization.tex").write_text("\n".join(tex))

summary_lines=[
 "# NiyamTrace-X Wave 4 Q1 evidence",
 "",
 "## Experiment A — external stateful authorization replay",
 "- Clean completion: **"+format(ntx.clean_completion,".2%")+"**",
 "- Unsafe commit: **"+format(ntx.unsafe_commit_rate,".2%")+"**",
 "- Attack opportunities: **"+str(int(ntx.attack_n))+"**",
 "",
 "## Experiment B — frozen post-v3 unseen red team",
 "- Attack recall: **"+format(rb.attack_recall,".2%")+"**",
 "- Clean FPR: **"+format(rb.clean_fpr,".2%")+"**",
 "- Residual misses: **"+str(int(rb.attack_missed))+"**",
 "",
 "## Experiment C — Broker V3",
 "- Sequential trials: **"+str(int(br.trials))+"**",
 "- Unsafe commits: **"+str(int(br.unsafe_commits))+"**",
 "- Race-safe rate: **"+format(br.race_safe_rate,".2%")+"**",
 "",
 "Do not remove the claim boundaries in Q1_claim_evidence_matrix.csv."
]
(ROOT/"Q1_EXPERIMENT_SUMMARY.md").write_text("\n".join(summary_lines))
print("\n".join(summary_lines))

In [ ]:
# Include optional human benchmark results if present, then package everything.
optional=BASE/"NTX_Q1_W4_D_HUMAN_BENCHMARK_RESULTS.zip"
if optional.exists():
    d=ROOT/"human_benchmark"
    d.mkdir()
    with zipfile.ZipFile(optional) as z:
        assert z.testzip() is None
        z.extractall(d)

out=BASE/"NiyamTrace-X_Q1_Wave4_Combined_Results.zip"
if out.exists():out.unlink()
with zipfile.ZipFile(out,"w",zipfile.ZIP_DEFLATED) as z:
    for p in ROOT.rglob("*"):
        if p.is_file():z.write(p,p.relative_to(ROOT))
    for name in required:
        z.write(BASE/name,Path("source_result_zips")/name)
    if optional.exists():
        z.write(optional,Path("source_result_zips")/optional.name)
print("Created:",out)
try:
    from google.colab import files
    files.download(str(out))
except Exception:pass